In [4]:
# ================================
# AGILITY LADDER ANALYSIS PIPELINE (FULL BIO VERSION)
# ================================

import cv2
import numpy as np
import mediapipe as mp
from scipy.signal import find_peaks
import math
import csv
import ctypes
from ultralytics import YOLO

In [5]:
# -------------------------------
# MediaPipe Setup
# -------------------------------
mpPose = mp.solutions.pose
pose = mpPose.Pose(min_detection_confidence=0.6)

In [6]:
# -------------------------------
# Utility Functions
# -------------------------------
def safe_mean(arr, default=0.0):
    return sum(arr)/len(arr) if len(arr) > 0 else default

def safe_std(arr, default=0.0):
    return float(np.std(arr)) if len(arr) > 0 else default

In [7]:
def get_display_size(frame_width, frame_height, margin=100):
    try:
        user32 = ctypes.windll.user32
        screen_width = user32.GetSystemMetrics(0)
        screen_height = user32.GetSystemMetrics(1)
    except Exception:
        screen_width, screen_height = frame_width, frame_height

    max_width = max(screen_width - margin, 1)
    max_height = max(screen_height - margin, 1)
    scale = min(max_width / max(frame_width, 1), max_height / max(frame_height, 1))

    return max(1, int(frame_width * scale)), max(1, int(frame_height * scale))

In [8]:
# -------------------------------
# Ladder Detection
# -------------------------------
def fallback_ladder_boxes(left_line_x, right_line_x, num_boxes = 15):
    boxes_point = []
    gap = (right_line_x - left_line_x)/num_boxes

    for i in range(num_boxes):
        point = (round((left_line_x + i*gap), 2), round((left_line_x + (i+1)*gap), 2))
        boxes_point.append(point)

    return boxes_point
    

In [11]:
def detect_ladder_boxes(path, num_boxes=15):
    try:
        model = YOLO("best.pt")
    except:
        return None

    cap = cv2.VideoCapture(path)

    left_x, right_x = [], []
    frame_count = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        results = model(frame, conf=0.5, verbose=False)

        centers = []
        for box in results[0].boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = box.conf.item()
            cx = (x1 + x2) // 2
            centers.append(cx)
            
            # Draw rectangle on cone
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            # Optional: confidence label
            cv2.putText(frame, f"Cone {conf:.2f}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        if len(centers) >= 2:
            centers.sort()
            left_x.append(centers[0])
            right_x.append(centers[-1])
            
        if len(left_x) > 50:
            break

        cv2.namedWindow("Video", cv2.WINDOW_NORMAL)
        cv2.setWindowProperty("Video", cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)
        cv2.imshow("Video", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

    # Final fixed threshold
    left_avg = int(np.mean(left_x))
    right_avg = int(np.mean(right_x))
    # threshold_x = int((left_avg + right_avg) / 2)

    # print("Left cone X:", left_avg)
    # print("Right cone X:", right_avg)
    # print("Threshold X:", threshold_x)
    # return left_avg, right_avg
    
    boxes_point = []
    gap = (right_avg - left_avg)/num_boxes

    for i in range(num_boxes):
        point = (round((left_avg + i*gap), 2), round((left_avg + (i+1)*gap), 2))
        boxes_point.append(point)

    return boxes_point

In [12]:
path = "data/agility_ladder_1.mp4"
print(detect_ladder_boxes(path, 10))

Left cone X: 102
Right cone X: 686
[(102.0, 160.4), (160.4, 218.8), (218.8, 277.2), (277.2, 335.6), (335.6, 394.0), (394.0, 452.4), (452.4, 510.8), (510.8, 569.2), (569.2, 627.6), (627.6, 686.0)]


In [8]:
# -------------------------------
# Foot Tracking
# -------------------------------
def get_foot_positions(lms, frame_width, frame_height):
    left = (int(((lms[31].x + lms[31].x)/2)*frame_width), int(((lms[31].y + lms[31].y)/2)*frame_height))
    right = (int(((lms[30].x + lms[32].x)/2)*frame_width), int(((lms[30].y + lms[32].y)/2)*frame_height))
    return left, right

In [9]:
def calculate_foot_distance(lms, frame_width, frame_height):
    x1, y1 = int(((lms[31].x + lms[31].x)/2)*frame_width), int(((lms[31].y + lms[31].y)/2)*frame_height)
    x2, y2 = int(((lms[30].x + lms[32].x)/2)*frame_width), int(((lms[30].y + lms[32].y)/2)*frame_height)
    
    # Calculate Euclidean distance
    distance = math.sqrt((x2 - x1)**2 + (y2 - y1)**2)
    return distance

In [10]:
def inside_box(left, right, boxes, complete_boxes):
    lx, ly = left
    rx, ry = right
    # for (x1, x2, y1, y2) in boxes:
        # if x1 <= x <= x2 and y1 <= y <= y2:
            # return True
    count = 0
    for (x1, x2) in boxes:
        count += 1
        if x1 <= lx <= x2 and x1 <= rx <= x2:
            complete_boxes.add(count)
            return complete_boxes
    return complete_boxes

In [11]:
# -------------------------------
# NEW: TORSO STABILITY
# -------------------------------
def torso_stability(lms):
    l_sh, r_sh = lms[11], lms[12]
    l_hip, r_hip = lms[23], lms[24]

    sh_vec = (r_sh.x - l_sh.x, r_sh.y - l_sh.y)
    hip_vec = (r_hip.x - l_hip.x, r_hip.y - l_hip.y)

    dot = sh_vec[0]*hip_vec[0] + sh_vec[1]*hip_vec[1]
    mag1 = math.sqrt(sh_vec[0]**2 + sh_vec[1]**2)
    mag2 = math.sqrt(hip_vec[0]**2 + hip_vec[1]**2)

    if mag1 == 0 or mag2 == 0:
        return 0

    return math.degrees(math.acos(dot/(mag1*mag2)))

In [12]:
# -------------------------------
# UPDATED MOVEMENT METRICS
# -------------------------------
def compute_metrics(hip_x, hip_y, left_y, right_y, time_list):

    hip_x = np.array(hip_x)
    hip_y = np.array(hip_y)
    left_y = np.array(left_y)
    right_y = np.array(right_y)
    t = np.array(time_list)

    # -------- STEP RHYTHM (FEET BASED) --------
    peaks_l, _ = find_peaks(-left_y, distance=5)
    peaks_r, _ = find_peaks(-right_y, distance=5)

    if len(peaks_l) > 1:
        rhythm_l = np.diff(t[peaks_l])
    else:
        rhythm_l = [0.2]

    if len(peaks_r) > 1:
        rhythm_r = np.diff(t[peaks_r])
    else:
        rhythm_r = [0.2]

    rhythm_var = safe_std(list(rhythm_l) + list(rhythm_r), default=0.2)

    # -------- LIMB SYNCHRONIZATION --------
    min_len = min(len(left_y), len(right_y))
    sync_diff = safe_mean(np.abs(left_y[:min_len] - right_y[:min_len]))

    # -------- BALANCE --------
    sway = safe_std(hip_x)

    # -------- FLUIDITY (ACCEL + JERK) --------
    if len(t) > 3:
        vel = np.diff(hip_y) / np.diff(t)
        accel = np.diff(vel)
        jerk = np.diff(accel)
        accel_var = safe_std(accel, default=1.0)
        jerk_var = safe_std(jerk, default=1.0)
    else:
        accel_var = 1.0
        jerk_var = 1.0

    return rhythm_var, sync_diff, sway, accel_var, jerk_var

In [13]:
def cal_time_score(time_sec):
    if time_sec <= 6.5:
        return 5
    elif time_sec <= 7.5:
        return 4
    elif time_sec <= 8.5:
        return 3
    elif time_sec <= 9.5:
        return 2
    else:
        return 1

In [14]:
def error_adjustment(time_score, error_count):
    if error_count == 0:
        return time_score
    elif error_count <= 2:
        return time_score - 0.5
    elif error_count <= 5:
        return time_score - 1
    else:
        return min(2, time_score)

In [15]:
# -------------------------------
# SCORING FUNCTIONS
# -------------------------------
def balance_score(sway):
    if sway < 0.02: return 2
    elif sway < 0.05: return 1
    else: return 0

In [16]:
def sync_score(sync_diff):
    if sync_diff < 0.02: return 1
    elif sync_diff < 0.05: return 0.5
    else: return 0

In [17]:
def rhythm_score(rhythm):
    if rhythm < 0.05: return 1
    elif rhythm < 0.08: return 0.5
    else: return 0

In [18]:
def torso_score(avg_angle):
    if avg_angle < 20: return 2
    elif avg_angle < 30: return 1
    else: return 0

In [19]:
def fluidity_score(accel_var, jerk_var):
    if accel_var < 0.04 and jerk_var < 0.1: return 2
    elif accel_var < 0.08: return 1
    else: return 0

In [20]:
def final_score_calculation(time_score, mabc):
    if mabc >= 6:
        return time_score
    elif mabc >= 5:
        return time_score-1
    elif mabc >= 3:
        return time_score - 2
    else:
        return min(time_score, 2)

In [21]:
# -------------------------------
# MAIN PIPELINE
# -------------------------------
def agility_ladder(ID="ID001", name="Test", path="video.mp4", num_boxes = 10, save_csv=True):

    cap = cv2.VideoCapture(path)

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_width = int(cap.get(3))
    frame_height = int(cap.get(4))
    display_width, display_height = get_display_size(frame_width, frame_height)

    # ladder_boxes = [(525, 625), (625, 725), (725, 825), (825, 920), (920, 1020), (1020, 1100), (1100, 1190), (1190, 1275)]
    ladder_boxes = detect_ladder_boxes(path, num_boxes)
    # print(ladder_boxes)
    if ladder_boxes is None:
        print("Fallback_ladder")
        ladder_boxes = fallback_ladder_boxes(frame_width, frame_height, num_boxes)

    hip_x, hip_y, time_list = [], [], []
    left_y, right_y = [], []
    torso_angles = []
    time_frame_count = 0
    time_count_flag = False

    left_line_x = ladder_boxes[0][0]
    right_line_x = ladder_boxes[-1][1]
    print(left_line_x, right_line_x)
    thresold_dist = int((right_line_x-left_line_x)/(num_boxes*2))
    complete_boxes = set()
    box_flag = True

    error_count = 0
    frame_idx = 0
    window_name = "Agility Ladder Processing"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(window_name, display_width, display_height)

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if time_count_flag: 
            time_frame_count += 1

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(rgb)

        if results.pose_landmarks:
            lms = results.pose_landmarks.landmark

            # -------- FEET --------
            left, right = get_foot_positions(lms, frame_width, frame_height)
            cv2.line(frame, left, right, (0, 255, 0), 3)

            # time_frame_count
            x1, y1 = left
            x2, y2 = right
            if not time_count_flag and x1 >= left_line_x or x2 >= left_line_x:
                time_count_flag = True
            if x1 >= right_line_x and x2 >= right_line_x:
                time_count_flag = False
                break
            
            # condition for min dist of foot for box count
            foot_dist = calculate_foot_distance(lms, frame_width, frame_height)
            if time_count_flag and box_flag and foot_dist <= thresold_dist:
                complete_boxes =  inside_box(left, right, ladder_boxes, complete_boxes)
                cv2.circle(frame, (50,50), 12, (0, 0, 255), -1)

                

            left_y.append(lms[31].y)
            right_y.append(lms[32].y)

            # -------- HIP --------
            hx = (lms[23].x + lms[24].x) / 2
            hy = (lms[23].y + lms[24].y) / 2

            hip_x.append(hx)
            hip_y.append(hy)
            time_list.append(frame_idx / fps if fps else 0)

            # -------- TORSO --------
            torso_angles.append(torso_stability(lms))

            cv2.circle(frame, left, 6, (0, 255, 0), -1)
            cv2.circle(frame, right, 6, (0, 255, 0), -1)

        # for (x1, x2, y1, y2) in ladder_boxes:
        #     cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)
            for (x1, x2) in ladder_boxes:
                x1, x2 = int(x1), int(x2)
                cv2.rectangle(frame, (x1, 300), (x2, 400), (255, 0, 0), 2)

        cv2.putText(frame, f"Frame: {frame_idx}  Errors: {error_count} Box complete: {complete_boxes}", (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2,)
        cv2.imshow(window_name, frame)
        if cv2.waitKey(20) & 0xFF == ord("q"):
            break

        frame_idx += 1

    cap.release()
    cv2.destroyAllWindows()

    total_time = time_frame_count / fps if fps else 0
    time_score = cal_time_score(total_time)
    error_count = num_boxes - len(complete_boxes)
    time_score = error_adjustment(time_score, error_count)

    # ---------------- METRICS ----------------
    rhythm_var, sync_diff, sway, accel_var, jerk_var = compute_metrics(hip_x, hip_y, left_y, right_y, time_list)

    avg_torso = safe_mean(torso_angles)

    # ---------------- SCORES ----------------
    bal = balance_score(sway)
    syn = sync_score(sync_diff)
    rhy = rhythm_score(rhythm_var)
    flu = fluidity_score(accel_var, jerk_var)
    tor = torso_score(avg_torso)

    limb = syn + rhy
    mabc = bal + limb + flu + tor  # expanded model

    final_likert_score = final_score_calculation(time_score, mabc)
    if final_likert_score <= 0:
        final_likert_score = 0

    # # ---------------- FINAL ----------------
    # print("------ AGILITY LADDER RESULT ------")
    # time_score
    print(f"Total Time: {total_time:.2f} and frame count {time_frame_count}")
    # # error
    print(f"Box Complete: {len(complete_boxes)} and Errors: {error_count}")
    # balance
    print(f"Balance_Sway: {sway:.3f}")
    # Limb_syn
    print(f"Limb_Sync: {sync_diff:.3f}")
    print(f"Rhythm: {rhythm_var:.3f}")
    # directional_accuracy
    print(f"Directional Torso Angle: {avg_torso:.2f}")
    # movement fluidity
    print(f"Fluidity Accel Var: {accel_var:.3f}, Jerk Var: {jerk_var:.3f}")

    # score
    print("========================================")
    print(f"Time_score: {time_score}")
    print(f"MABC Score: {mabc}")
    print(f"final_score_calculation: {final_likert_score}")

    return final_likert_score

In [24]:
path = "data/agility_ladder_1.mp4"
agility_ladder(ID="ID001", name="Test", path=path, num_boxes = 10, save_csv=True)

Left cone X: 102
Right cone X: 686
102.0 686.0
Total Time: 12.63 and frame count 378
Box Complete: 8 and Errors: 2
Balance_Sway: 0.219
Limb_Sync: 0.020
Rhythm: 0.136
Directional Torso Angle: 12.09
Fluidity Accel Var: 0.068, Jerk Var: 0.099
Time_score: 0.5
MABC Score: 4
final_score_calculation: 0


0